#ChargeGrid Intelligence - GoodWe
###Solução Comercial

## ⚡ Simulação de Orquestração de Energia:

Este código simula um sistema de gestão de energia (EMS) para uma galeria comercial equipada com painéis solares, sistemas de armazenamento (BESS) e carregadores de veículos elétricos (EVs). O objetivo é otimizar o fluxo energético, maximizando o autoconsumo solar e minimizando custos operacionais.

### Fases Operacionais

| Fase | Período | Característica Principal |
| :--- | :--- | :--- |
| Pré-op | Manhã | Carregamento de baterias e preparação da demanda |
| Operacional | Dia | Geração solar ativa, carga de EVs e suporte à rede |
| Noturno | Noite | Descarga de baterias para suprir carga base |

### Hierarquia de Prioridades de Despacho

| Prioridade | Destino da Energia | Objetivo |
| :--- | :--- | :--- |
| 1 | Carga Crítica | Garantir continuidade da galeria |
| 2 | Carregadores EV | Atender demanda de mobilidade |
| 3 | Baterias (BESS) | Armazenar excedente solar |
| 4 | Rede Elétrica | Exportar excedente (venda) |

### Resultados da Simulação
O código processa séries temporais de irradiação solar e demanda de carga, retornando:
1. **Perfil de Carga:** Gráficos de consumo vs. geração.
2. **Estado de Carga (SoC):** Evolução do nível das baterias.
3. **KPIs Financeiros:** Economia gerada e eficiência do sistema.

In [ ]:
import math

def simular_orquestracao():
    # 1. ENTRADA DE DADOS
    metragem_galeria = float(input("Metragem total da galeria (m²): "))
    area_paineis = float(input("Área disponível para painéis (m²): "))

    qtd_paineis = math.floor(area_paineis / 2.256)
    potencia_total = (qtd_paineis * 450) / 1000
    geracao_pico = potencia_total * 0.72
    max_vagas = math.floor((geracao_pico * 0.6) / 7)

    print(f"\nPotência instalada: {potencia_total:.2f} kWp")
    print(f"Máximo de vagas suportadas: {max_vagas}")
    vagas_desejadas = int(input("Número de vagas desejadas: "))

    area_bateria = float(input("Metragem disponível para sala de baterias (m²): "))
    qtd_baterias = math.floor(area_bateria / 0.4)
    capacidade_total_bateria = qtd_baterias * 15
    carga_bateria = 10.0

    consumo_diario = (metragem_galeria * 25) / 365

    # Dados de geração (base 4.5kWp, escalonado pela potência real)
    geracao_base = [0.5, 1.2, 2.0, 2.8, 2.5, 3.8, 4.5, 4.8, 4.7, 4.2, 3.2, 1.8, 0.5, 0, 0, 0, 0]
    percentuais_galeria = {10: 0.03, 11: 0.03, 12: 0.05, 13: 0.05, 14: 0.045, 15: 0.045, 16: 0.045, 17: 0.04, 18: 0.04, 19: 0.02, 20: 0.02, 21: 0.02, 22: 0.02}

    resultados = []
    totais = {"gerado": 0, "galeria": 0, "ev": 0, "galeria_solar": 0, "armazenado": 0, "usado": 0, "rede": 0, "custo": 0}

    print("\n" + "="*120)
    print(f"{'Hora':<6} | {'Status':<15} | {'EVs':<5} | {'Solar':<6} | {'Galeria':<8} | {'Bateria Arm':<11} | {'Bateria Usa':<11} | {'Rede':<6} | {'Custo':<8}")

    for hora in range(6, 23):
        solar = (geracao_base[hora-6] / 4.5) * potencia_total
        cons_galeria = consumo_diario * percentuais_galeria.get(hora, 0)
        carros = 0
        if 10 <= hora < 19: carros = int(input(f"Carros às {hora}h: "))
        elif hora >= 19: carros = int(input(f"Carros à noite às {hora}h: "))

        status = "Pré-op" if hora < 10 else ("Operacional" if hora < 19 else "Noturno")
        tarifa = 1.5 if hora >= 17 else (1.0 if hora >= 10 else 0.85)

        b_armazenado, b_usado, rede, galeria_solar = 0, 0, 0, 0

        if status == "Pré-operação":
            espaco = capacidade_total_bateria - carga_bateria
            b_armazenado = min(solar, espaco)
            carga_bateria += b_armazenado
        elif status == "Operacional":
            demanda_ev = carros * 7
            disponivel = solar
            # Prioridade 1: EVs
            energia_ev = min(disponivel, demanda_ev)
            disponivel -= energia_ev
            # Prioridade 2: Bateria
            espaco = capacidade_total_bateria - carga_bateria
            b_armazenado = min(disponivel, espaco)
            carga_bateria += b_armazenado
            disponivel -= b_armazenado
            # Prioridade 4: Galeria Solar
            if carga_bateria >= capacidade_total_bateria:
                galeria_solar = min(disponivel, cons_galeria)
            rede = (demanda_ev - energia_ev) + (cons_galeria - galeria_solar)
        else:
            demanda_total = (carros * 7) + cons_galeria
            b_usado = min(carga_bateria, demanda_total)
            carga_bateria -= b_usado
            rede = max(0, demanda_total - b_usado)

        custo = rede * tarifa
        totais.update({"gerado": totais["gerado"]+solar, "galeria": totais["galeria"]+cons_galeria, "ev": totais["ev"]+(carros*7), "galeria_solar": totais["galeria_solar"]+galeria_solar, "armazenado": totais["armazenado"]+b_armazenado, "usado": totais["usado"]+b_usado, "rede": totais["rede"]+rede, "custo": totais["custo"]+custo})
        print(f"{hora:02}h   | {status:<15} | {carros:<5} | {solar:<6.2f} | {cons_galeria:<8.2f} | {b_armazenado:<11.2f} | {b_usado:<11.2f} | {rede:<6.2f} | R${custo:<6.2f}")

    print("\nRESUMO FINAL")
    print(f"Total Gerado: {totais['gerado']:.2f} kWh | Custo Total: R${totais['custo']:.2f}")

simular_orquestracao()

Metragem total da galeria (m²): 400
Área disponível para painéis (m²): 350

Potência instalada: 69.75 kWp
Máximo de vagas suportadas: 4
Número de vagas desejadas: 4
Metragem disponível para sala de baterias (m²): 50

Hora   | Status          | EVs   | Solar  | Galeria  | Bateria Arm | Bateria Usa | Rede   | Custo   
06h   | Pré-op          | 0     | 7.75   | 0.00     | 7.75        | 0.00        | 0.00   | R$0.00  
07h   | Pré-op          | 0     | 18.60  | 0.00     | 18.60       | 0.00        | 0.00   | R$0.00  
08h   | Pré-op          | 0     | 31.00  | 0.00     | 31.00       | 0.00        | 0.00   | R$0.00  
09h   | Pré-op          | 0     | 43.40  | 0.00     | 43.40       | 0.00        | 0.00   | R$0.00  
Carros às 10h: 2
10h   | Operacional     | 2     | 38.75  | 0.82     | 24.75       | 0.00        | 0.82   | R$0.82  
Carros às 11h: 3
11h   | Operacional     | 3     | 58.90  | 0.82     | 37.90       | 0.00        | 0.82   | R$0.82  
Carros às 12h: 3
12h   | Operacional     | 3    

# 🌡️ Economia em caso de Projeto de Conforto Térmico

## Simulação de Eficiência Energética e Conforto Térmico

Este código realiza uma análise financeira e técnica para otimizar o consumo energético em galerias comerciais. O objetivo é equilibrar o conforto térmico dos ocupantes com a viabilidade econômica de tecnologias sustentáveis.

## Potencial de Redução de Carga Térmica

| Medida | Redução Estimada |
| :--- | :--- |
| Isolamento Térmico | 15% |
| Vidros de Controle Solar | 20% |
| Sombreamento Externo | 25% |
| Ar-Condicionado Eficiente | 30% |
| Automação Inteligente | 10% |

## Resultados da Simulação

O script processa os dados de entrada e retorna:
- **Economia Anual:** Valor monetário economizado em custos operacionais.
- **Economia em 10 Anos:** Projeção do retorno sobre o investimento (ROI).
- **Vagas Adicionais:** Capacidade de carregamento de veículos elétricos viabilizada pelo excedente energético.

In [ ]:
def calcular_conforto_termico(temperatura, umidade, velocidade_ar):
    # Constantes para o cálculo (simplificado)
    # PMV (Predicted Mean Vote) - Modelo de Fanger
    # Este é um exemplo didático de cálculo de conforto térmico

    # Ajuste de temperatura baseada na umidade e velocidade do ar
    temperatura_efetiva = temperatura - (0.05 * umidade) + (0.5 * velocidade_ar)

    if 18 <= temperatura_efetiva <= 24:
        status = "Confortável"
    elif temperatura_efetiva < 18:
        status = "Frio"
    else:
        status = "Quente"

    return temperatura_efetiva, status

# Dados de entrada
medicoes = [
    {"temp": 22, "umidade": 50, "vel": 0.2},
    {"temp": 15, "umidade": 40, "vel": 0.1},
    {"temp": 28, "umidade": 60, "vel": 0.5}
]

print("--- Resumo de Conforto Térmico ---")
for i, m in enumerate(medicoes):
    temp_efetiva, status = calcular_conforto_termico(m['temp'], m['umidade'], m['vel'])
    print(f"Medição {i+1}: Temp Efetiva = {temp_efetiva:.2f}°C | Status: {status}")

print("Cálculo finalizado com sucesso.")

--- Resumo de Conforto Térmico ---
Medição 1: Temp Efetiva = 19.60°C | Status: Confortável
Medição 2: Temp Efetiva = 13.05°C | Status: Frio
Medição 3: Temp Efetiva = 25.25°C | Status: Quente
Cálculo finalizado com sucesso.


# 🔄 Carregamento EV - KM em kWh

## Simulação de Custos de Carregamento de Veículos Elétricos (EV)

Este script realiza a conversão precisa de quilometragem percorrida para consumo de energia (kWh) e estima os custos operacionais, permitindo uma análise comparativa direta entre veículos elétricos e a combustão.

### Modelos Suportados

| Modelo | Consumo Médio (kWh/km) |
| :--- | :--- |
| Tesla Model 3 | 0.15 |
| Tesla Model Y | 0.17 |
| Nissan Leaf | 0.16 |
| Chevrolet Bolt | 0.16 |
| BMW i3 | 0.14 |

### Parâmetros e Fórmulas

| Parâmetro | Fórmula / Descrição |
| :--- | :--- |
| Consumo (kWh) | Distância (km) × Consumo Médio (kWh/km) |
| Custo Carregamento | Consumo (kWh) × Tarifa de Energia (R$/kWh) |
| Economia vs Gasolina | (Custo Gasolina) - (Custo EV) |
| % de Economia | (Economia / Custo Gasolina) × 100 |

### Resultados Calculados
O código processa os dados de entrada e retorna:
1. **Consumo Total:** Energia necessária em kWh.
2. **Custo de Carregamento:** Valor financeiro gasto na recarga.
3. **Comparativo:** Economia absoluta em relação a um veículo a combustão.
4. **Eficiência:** Percentual de redução de custos operacionais.

In [ ]:
def carregar_ev():
    modelos = {
        'Tesla Model 3': 0.15,
        'Tesla Model Y': 0.17,
        'Nissan Leaf': 0.16,
        'Chevrolet Bolt': 0.16,
        'BMW i3': 0.14
    }

    print('Selecione o modelo do veículo:')
    for modelo in modelos.keys():
        print(f'- {modelo}')
    print('- Outro')

    escolha = input('Digite o modelo do veículo: ')

    if escolha in modelos:
        consumo_medio = modelos[escolha]
        print(f'Consumo sugerido para {escolha}: {consumo_medio} kWh/km')
    elif escolha == 'Outro':
        consumo_medio = float(input('Digite o consumo médio manualmente (kWh/km): '))
    else:
        print('Modelo não reconhecido. Por favor, insira o consumo manualmente.')
        consumo_medio = float(input('Digite o consumo médio (kWh/km): '))

    distancia = float(input('Digite a distância a percorrer (km): '))
    energia_necessaria = distancia * consumo_medio
    print(f'Energia necessária: {energia_necessaria:.2f} kWh')

if __name__ == '__main__':
    carregar_ev()

Selecione o modelo do veículo:
- Tesla Model 3
- Tesla Model Y
- Nissan Leaf
- Chevrolet Bolt
- BMW i3
- Outro
Digite o modelo do veículo: Tesla Model 3
Consumo sugerido para Tesla Model 3: 0.15 kWh/km
Digite a distância a percorrer (km): 10
Energia necessária: 1.50 kWh


# 🌱 Simulação de Emissão de Carbono: EV vs. Gasolina

Este código realiza uma análise comparativa entre veículos elétricos (EV) e veículos a combustão (gasolina), calculando o impacto ambiental com base na quilometragem percorrida e nos fatores de emissão específicos do Brasil.

### 1. Fatores de Emissão Utilizados

| Parâmetro | Valor/Unidade |
| :--- | :--- |
| Fator Emissão Brasil (Grid) | 0.06 kg CO2/kWh |
| Fator Emissão Gasolina | 0.12 kg CO2/km |
| Absorção por Árvore | 22 kg CO2/ano |
| Equivalência Voo | 0.25 kg CO2/km |

### 2. Parâmetros e Fórmulas de Cálculo

| Cálculo | Fórmula |
| :--- | :--- |
| Emissão EV | (km / consumo_km_kwh) * fator_grid |
| Emissão Gasolina | km * fator_gasolina |
| Redução | Emissão Gasolina - Emissão EV |
| % Redução | (Redução / Emissão Gasolina) * 100 |

### O que o código calcula:
O script processa os dados de entrada e retorna:
* **🌍 Emissão Total (kg CO2):** Comparativo direto entre os dois modais.
* **📉  Redução de Emissão:** O ganho ambiental líquido ao optar pelo EV.
* **🌱 Equivalência em Árvores:** Quantidade de árvores necessárias para compensar a emissão evitada.
* **✈️ Equivalência em Voo:** Distância aérea equivalente ao impacto evitado.

In [ ]:
def calcular_emissao_carbono():
    # Configurações de constantes
    FATOR_EMISSAO_BRASIL_KWH = 0.084
    FATOR_EMISSAO_GASOLINA_KM = 0.231
    ARVORES_POR_KG_CO2 = 0.05  # Estimativa: 1 árvore absorve aprox 20kg/ano, ~1.6kg/mes
    KM_VOO_POR_KG_CO2 = 2.0    # Estimativa simplificada

    # 1. Entrada interativa
    try:
        consumo_medio = float(input("Digite o consumo médio (kWh/km) [padrão 0.15]: ") or 0.15)
        quilometragens_input = input("Digite as quilometragens separadas por vírgula (ex: 100, 150, 200): ")
        lista_quilometragens = [float(km.strip()) for km in quilometragens_input.split(",")]
    except ValueError:
        print("Erro: Entrada inválida. Certifique-se de usar números.")
        return

    # Cabeçalho da tabela
    print("\n╔" + "═"*80 + "╗")
    print("║ KM      ║ kWh     ║ Emiss. EV (kg) ║ Emiss. Gas (kg) ║ Redução (kg) ║ % Redução ║")
    print("╠" + "═"*80 + "╣")

    total_km = 0
    total_kwh = 0
    total_reducao = 0
    total_emissao_gas = 0

    # 3. Cálculos e 4. Tabela
    for km in lista_quilometragens:
        energia_necessaria = km * consumo_medio
        emissao_ev = energia_necessaria * FATOR_EMISSAO_BRASIL_KWH
        emissao_gasolina = km * FATOR_EMISSAO_GASOLINA_KM
        reducao = emissao_gasolina - emissao_ev
        percentual_reducao = (reducao / emissao_gasolina) * 100 if emissao_gasolina > 0 else 0

        total_km += km
        total_kwh += energia_necessaria
        total_reducao += reducao
        total_emissao_gas += emissao_gasolina

        print(f"║ {km:<7.1f} ║ {energia_necessaria:<7.2f} ║ {emissao_ev:<14.2f} ║ {emissao_gasolina:<15.2f} ║ {reducao:<12.2f} ║ {percentual_reducao:<9.1f} ║")

    print("╚" + "═"*80 + "╝")

    # 5. Resumo Executivo
    percentual_total = (total_reducao / total_emissao_gas) * 100 if total_emissao_gas > 0 else 0
    arvores = total_reducao * ARVORES_POR_KG_CO2
    voos = total_reducao * KM_VOO_POR_KG_CO2

    print("\nRESUMO EXECUTIVO")
    print(f"⚡ Consumo: {consumo_medio:.2f} kWh/km")
    print(f"🌍 Total: {total_km:.1f} km | {total_kwh:.2f} kWh")
    print(f"📉 Redução de Emissão: {total_reducao:.2f} kg CO2 ({percentual_total:.1f}%)")
    print(f"🌱 Equivalência: {arvores:.2f} árvores plantadas (aprox. 1 mês cada)")
    print(f"✈️ Equivalência: {voos:.2f} km de voo evitados")

if __name__ == "__main__":
    calcular_emissao_carbono()

Digite o consumo médio (kWh/km) [padrão 0.15]: 0.15
Digite as quilometragens separadas por vírgula (ex: 100, 150, 200): 100,150

╔════════════════════════════════════════════════════════════════════════════════╗
║ KM      ║ kWh     ║ Emiss. EV (kg) ║ Emiss. Gas (kg) ║ Redução (kg) ║ % Redução ║
╠════════════════════════════════════════════════════════════════════════════════╣
║ 100.0   ║ 15.00   ║ 1.26           ║ 23.10           ║ 21.84        ║ 94.5      ║
║ 150.0   ║ 22.50   ║ 1.89           ║ 34.65           ║ 32.76        ║ 94.5      ║
╚════════════════════════════════════════════════════════════════════════════════╝

RESUMO EXECUTIVO
⚡ Consumo: 0.15 kWh/km
🌍 Total: 250.0 km | 37.50 kWh
📉 Redução de Emissão: 54.60 kg CO2 (94.5%)
🌱 Equivalência: 2.73 árvores plantadas (aprox. 1 mês cada)
✈️ Equivalência: 109.20 km de voo evitados


#BIPV em Fachadas (Premium) - A desenvolver*

## Gerar mais energia limpa --> Maximizar a geração fotovoltaica.
* Instalar painéis fotovoltaicos em fachadas pode aumentar a geração de energia, mas também pode aumentar o aquecimento da envoltória do edifício. Se o calor adicional aumentar muito o consumo do ar-condicionado (HVAC), parte da energia gerada será "gasta" para resfriar o edifício.

* O objetivo deve ser **maximizar o benefício energético líquido**, além de **maximizar a geração fotovoltaica**.

* Evitar que a Fachada Aqueça e aumente o consumo de HVAC (um dos principais gastos de energia da galeria) : Fachada dupla

Qual combinação de fachada norte, fachada oeste, cobertura fotovoltaica e baterias maximiza o benefício energético e econômico de uma galeria comercial?

###Entradas

O usuário informa:

* Quantidade de fachadas  
* Orientação de cada fachada  
* Área de cada fachada  
* Capacidade da bateria (kWh)  
* Existência ou intensão de fachada ventilada
* U-Value da envoltória  
* Consumo diário da galeria    


###Saídas

Para cada hora:

* Geração fotovoltaica  
* Consumo da galeria  
* Consumo HVAC  
* Saldo energético  
* Nível da bateria

###Beneficio=EFV​+EHVAC_evitado​

###Criar Dataset a partir do EnergyPlus



